# Manage hosted agent sessions

A hosted agent session is a sandbox with a persisted file system (`$HOME`) tied to one logical
workload, such as one user's chat. The platform creates a session on the first call that names
none, keeps `$HOME` across turns and idle periods, and deletes the session after 30 days without
activity. This notebook works the session API against the `contoso-rm-notes-agent` deployed by
[08-12-01](08-12-01-deploy-hosted-agent-from-code.ipynb), whose notes live in `$HOME`, so every
session operation has a visible effect.

Sessions are not conversations. A conversation (`previous_response_id` or a `conversation` ID)
is the message history the platform stores for the Responses protocol. A session is the compute
and the files. Passing `agent_session_id` on a call binds it to a sandbox; it does not replay
earlier messages. See [`08-12-00-hosted-agent-sessions.md`](08-12-00-hosted-agent-sessions.md).

**What this notebook does:**
1. Lets the platform create a session, then continues in it with `agent_session_id` and `previous_response_id`
2. Shows that a call without a session ID lands in a fresh sandbox with no notes
3. Creates a session explicitly, uploads a file before the first turn, and downloads a file the agent wrote
4. Lists the agent's sessions across users, tallied by status
5. Watches a session go idle after the version's 120-second timeout and resume with its notes intact
6. Stops a session (compute off, files kept) and resumes it
7. Deletes a session (files gone) and shows what a call with its ID does afterwards

## Prerequisites

1. **08-12-01** has run: `contoso-rm-notes-agent` exists on the Team Alpha project with an active version.
2. **`.env` file**: `ALPHA_FOUNDRY_PROJECT_ENDPOINT` (set by the project spoke deployment).
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: by default a caller sees only the sessions its own identity created. Listing,
   reading and deleting every session on the agent (Step 5) needs the **Foundry User** role on the
   project, which carries the `sessions/read` and `sessions/write` data actions.

## Step 1: Configuration

In [1]:
import os, subprocess, time
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv
import openai
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import VersionRefIndicator
from azure.core.exceptions import ResourceNotFoundError

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
AGENT_NAME       = "contoso-rm-notes-agent"
DATA_DIR         = Path("data")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

agent  = client.agents.get(agent_name=AGENT_NAME)
latest = agent.versions.latest
AGENT_VERSION = latest.version
IDLE_TIMEOUT  = latest.definition["session_configuration"]["idle_timeout_seconds"]

def label(status) -> str:
    """The SDK returns known statuses as enum members; print their value."""
    return getattr(status, "value", status)

print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"Agent:            {AGENT_NAME} v{AGENT_VERSION} ({label(latest.status)})")
print(f"Dependencies:     {latest.definition['code_configuration']['dependency_resolution']}")
print(f"Idle timeout:     {IDLE_TIMEOUT}s")

Project Endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent:            contoso-rm-notes-agent v2 (active)
Dependencies:     bundled
Idle timeout:     120s


In [2]:
agent_client = client.get_openai_client(agent_name=AGENT_NAME)

def ask(question: str, session_id: str | None = None, previous_response_id: str | None = None, retries: int = 6):
    """One Responses call to the agent endpoint. session_id binds it to a sandbox;
    previous_response_id threads the conversation. They are independent."""
    extra = {"agent_session_id": session_id} if session_id else {}
    # An explicit previous_response_id=None is sent as null and rejected with 400, so omit it.
    chain = {"previous_response_id": previous_response_id} if previous_response_id else {}
    for attempt in range(1, retries + 1):
        try:
            response = agent_client.responses.create(input=question, extra_body=extra, **chain)
            break
        except openai.APIStatusError as e:
            # 424 session_not_ready and 5xx while a sandbox starts
            if e.status_code not in (424, 500, 502, 503) or attempt == retries:
                raise
            print(f"   HTTP {e.status_code} (attempt {attempt}/{retries}), retrying in 20s...")
            time.sleep(20)
    if response.status != "completed":
        raise RuntimeError(f"response {response.status}: {response.error}")
    print(f"\U0001f9d1 {question}\n\U0001f916 {response.output_text}")
    print(f"   agent_session_id={response.model_extra.get('agent_session_id')}  response.id={response.id}\n")
    return response

def when(dt):
    return dt.strftime("%H:%M:%S") if dt else "-"

def show_session(s, mark=""):
    print(f"  {s.agent_session_id}  {label(s.status):<8}  created={when(s.created_at)}  "
          f"last_accessed={when(s.last_accessed_at)}  stopped={when(s.stopped_at)}  "
          f"expires={s.expires_at:%Y-%m-%d}  v{s.version_indicator.agent_version}{mark}")

def get_session(session_id):
    show_session(client.agents.get_session(agent_name=AGENT_NAME, session_id=session_id))

## Step 2: Let the platform create the session, then stay in it

A call without `agent_session_id` is routed to a sandbox the platform creates for it, and the
response carries its ID. To keep using that sandbox, send the ID back in the body on later calls.
To keep the conversation, chain `previous_response_id` as usual. The second call below does both,
so the agent remembers the first turn and finds the note it saved.

In [3]:
first = ask("Save a note titled 'Meyer family review' with this body: equity allocation drifted above "
            "the strategic range; the clients want no change to the risk profile; they asked for a "
            "comparison of Contoso Sustainable Income and Contoso Income, in German.")
SESSION_A = first.model_extra["agent_session_id"]

second = ask("List my notes. Which note did you just save for me?",
             session_id=SESSION_A, previous_response_id=first.id)

🧑 Save a note titled 'Meyer family review' with this body: equity allocation drifted above the strategic range; the clients want no change to the risk profile; they asked for a comparison of Contoso Sustainable Income and Contoso Income, in German.
🤖 Saved as `notes/meyer-family-review.md`.
   agent_session_id={session-id}  response.id=caresp_049db47afbe9e248006OQWsEAZyrlUQwKmXyRjE6SlQL9TB3Co



🧑 List my notes. Which note did you just save for me?
🤖 You have 1 note:

- `notes/meyer-family-review.md` — **Meyer family review**

I just saved: **Meyer family review**.
   agent_session_id={session-id}  response.id=caresp_049db47afbe9e24800PcoGXsghzhFozFgBnXQvaxPE6bH63nIR



## Step 3: No session ID, new sandbox

Each session has its own `$HOME`. A call that names no session lands in a fresh sandbox, so
the agent finds no notes and `where_am_i` reports a different session ID.

In [4]:
other = ask("List my notes, then tell me where you are running.")
SESSION_B = other.model_extra["agent_session_id"]

print(f"SESSION_A = {SESSION_A}\nSESSION_B = {SESSION_B}\nsame sandbox: {SESSION_A == SESSION_B}")

🧑 List my notes, then tell me where you are running.
🤖 No notes in this session yet.

Running in:
- session_id=ba16395922db74380cd1e408726dc68ea012a3dd2b61315cee9dc1e10482daf
- host=adc-sandbox
- home=/home/session
   agent_session_id={session-id}  response.id=caresp_09008867dc22293a00FJYgyFM6jvk2j8gMJBsIkPGjwDF57xEs

SESSION_A = e49b1f8c819c9ccab9bc76d5a0632743e35c187ed6f56f30b77a032732043cc
SESSION_B = ba16395922db74380cd1e408726dc68ea012a3dd2b61315cee9dc1e10482daf
same sandbox: False


## Step 4: Create a session first, upload a file, read back what the agent wrote

Creating a session explicitly is for the cases where the sandbox must exist before the first turn:
uploading files to it, or pinning it to a specific agent version with `version_indicator`, which
later turns keep using even after a newer version ships. Files uploaded through the session files
API land in `$HOME` (up to 50 MB each), and files the container writes under `$HOME` are visible
through the same API, so the note the agent saves can be listed and downloaded.

In [5]:
created = client.agents.create_session(
    agent_name=AGENT_NAME,
    version_indicator=VersionRefIndicator(agent_version=AGENT_VERSION),
)
SESSION_C = created.agent_session_id
show_session(created)

agenda = DATA_DIR / "meeting-agenda.md"
result = client.agents.upload_session_file(
    agent_name=AGENT_NAME, session_id=SESSION_C, content=agenda.read_bytes(), path=agenda.name,
)
print(f"\nuploaded {result.path} ({result.bytes_written} bytes)")

print("\nfiles in the session home directory:")
for entry in client.agents.list_session_files(agent_name=AGENT_NAME, session_id=SESSION_C):
    print(f"  {entry.name:25s} {'dir' if entry.is_directory else f'{entry.size} bytes'}")

  {session-id}  active    created=14:38:49  last_accessed=14:38:49  stopped=-  expires=2026-11-10  v2



uploaded meeting-agenda.md (1024 bytes)

files in the session home directory:


  meeting-agenda.md         1024 bytes


In [6]:
ask("Read meeting-agenda.md and save a note titled 'Meyer review actions' that lists only the action "
    "items with their owners and dates. Reply with the path the tool returned.", session_id=SESSION_C)

notes = list(client.agents.list_session_files(agent_name=AGENT_NAME, session_id=SESSION_C, path="notes"))
print("notes/ in the sandbox:")
for entry in notes:
    print(f"  {entry.name} ({entry.size} bytes)")

note_path = f"notes/{notes[0].name}"
content = b"".join(client.agents.download_session_file(agent_name=AGENT_NAME, session_id=SESSION_C, path=note_path))
local = DATA_DIR / "downloaded" / notes[0].name
local.parent.mkdir(exist_ok=True)
local.write_bytes(content)
print(f"\ndownloaded {note_path} to {local}:\n")
print(content.decode("utf-8"))

🧑 Read meeting-agenda.md and save a note titled 'Meyer review actions' that lists only the action items with their owners and dates. Reply with the path the tool returned.
🤖 notes/meyer-review-actions.md
   agent_session_id={session-id}  response.id=caresp_049dd610a6d6ef3500XBWGF7IhYgIQnY8PeFCtD6gBgoIA4po6



notes/ in the sandbox:
  meyer-review-actions.md (324 bytes)



downloaded notes/meyer-review-actions.md to data/downloaded/meyer-review-actions.md:

# Meyer review actions

Saved: 2026-10-11T14:39:00+00:00

- Lea Fischer — send the rebalancing proposal and the updated IPS draft — by 2026-10-21
- Marco Rossi — prepare a one-page comparison of Contoso Sustainable Income and Contoso Income
- Lea Fischer — book an onboarding meeting with Nina Meyer — for Q1 2027



## Step 5: List the agent's sessions

`list_sessions` returns the sessions on the agent endpoint. With the Foundry User role it covers
every caller's sessions, which is the view an operator needs for cost or incident work; without
it, only the caller's own. Statuses: `creating`, `active` (compute running), `idle` (compute
released, files kept), `failed`, `deleting`, `deleted`, `expired`. `expires_at` is a rolling
30 days from the last activity.

In [7]:
sessions = list(client.agents.list_sessions(agent_name=AGENT_NAME, limit=100))
mine = {SESSION_A: " <- A", SESSION_B: " <- B", SESSION_C: " <- C"}

print(f"{len(sessions)} sessions on {AGENT_NAME}, by status: {dict(Counter(label(s.status) for s in sessions))}\n")
for s in sorted(sessions, key=lambda s: s.created_at, reverse=True)[:12]:
    show_session(s, mine.get(s.agent_session_id, ""))

3 sessions on contoso-rm-notes-agent, by status: {'idle': 2, 'deleted': 1}

  748181e2365feef201f4d346374e612552b3fd638d73f48eb9c64b05eef4c68  deleted   created=14:34:37  last_accessed=14:34:37  stopped=-  expires=2026-11-10  v2
  b2c51cfbf132adb5ae902c24ccd8579fc74fe875567cf1c72536f3ad5089563  idle      created=14:32:25  last_accessed=14:32:25  stopped=14:30:17  expires=2026-11-10  v2
  208d968a95029ed723fbf41067dfe92502cf849ef22c7ef7181ae38e4f262d3  idle      created=14:32:19  last_accessed=14:32:19  stopped=14:29:55  expires=2026-11-10  v1


## Step 6: Idle timeout, then resume

The version's idle timeout is 120 seconds. Session A has not been used since Step 2. Once that
window passes after its last call, the platform releases its compute and the session turns
`idle`, with `stopped_at` recording when the sandbox stopped. The timeout is a lower bound: in
our runs `stopped_at` came a minute or more after the 120 seconds, and the listed status caught
up with it a while later, so the loop below polls for up to four times the timeout. The next call
with the session's ID provisions new compute, restores `$HOME`, and the agent still finds the
note from Step 2. That call pays a cold start.

In [8]:
print(f"Session A, idle timeout {IDLE_TIMEOUT}s:")
t0 = time.time()
while True:
    s = client.agents.get_session(agent_name=AGENT_NAME, session_id=SESSION_A)
    print(f"  {time.time() - t0:4.0f}s  {label(s.status)}")
    if s.status != "active" or time.time() - t0 > IDLE_TIMEOUT * 4:
        break
    time.sleep(20)
show_session(s)

print("\nResuming session A:")
t0 = time.time()
ask("List my notes, then tell me where you are running.", session_id=SESSION_A)
print(f"resume, including the cold start: {time.time() - t0:.0f}s")
get_session(SESSION_A)

Session A, idle timeout 120s:
     0s  active


    20s  active


    41s  active


    61s  active


    81s  active


...

   202s  active


   223s  active


   243s  active


   263s  idle
  e49b1f8c819c9ccab9bc76d5a0632743e35c187ed6f56f30b77a032732043cc  idle      created=14:37:59  last_accessed=14:43:13  stopped=14:41:25  expires=2026-11-10  v2

Resuming session A:


🧑 List my notes, then tell me where you are running.
🤖 Notes:
- notes/meyer-family-review.md

Running in:
- session_id=e49b1f8c819c9ccab9bc76d5a0632743e35c187ed6f56f30b77a032732043cc
- host=adc-sandbox
- home=/home/session
   agent_session_id={session-id}  response.id=caresp_08ac5e11dc32740500ceOvREkHKZHTXXBKC1rfr5FWtUWICflv

resume, including the cold start: 19s
  e49b1f8c819c9ccab9bc76d5a0632743e35c187ed6f56f30b77a032732043cc  active    created=14:37:59  last_accessed=14:43:30  stopped=-  expires=2026-11-10  v2


## Step 7: Stop a session

`stop_session` ends the running compute right away and keeps the persisted file system, so it is
an explicit, early version of what the idle timeout does. The session stays listed, and the next
call with its ID resumes it with its files. Stopping a session that is already stopped succeeds.

In [9]:
client.agents.stop_session(agent_name=AGENT_NAME, session_id=SESSION_C)
print("after stop_session:")
get_session(SESSION_C)

print("\nResuming session C:")
ask("List my notes.", session_id=SESSION_C)
get_session(SESSION_C)

after stop_session:
  {session-id}  idle      created=14:38:49  last_accessed=14:43:45  stopped=14:43:45  expires=2026-11-10  v2

Resuming session C:


🧑 List my notes.
🤖 - notes/meyer-review-actions.md
   agent_session_id={session-id}  response.id=caresp_0ac6f0ae28cf3f7800BGaLgouxDR0boV3wDPRYl5T7cQJdzUfX

  {session-id}  active    created=14:38:49  last_accessed=14:43:47  stopped=-  expires=2026-11-10  v2


## Step 8: Delete a session

`delete_session` terminates the sandbox and releases its storage, and `get_session` then returns
404 `session_not_found`. The notes are gone with it. A Responses call that still carries the
deleted ID does not fail, though: session IDs may be caller-provided, so the platform starts a
fresh, empty sandbox under that ID and the agent reports no notes. An agent cannot be deleted
while it has sessions unless the delete is forced.

In [10]:
client.agents.delete_session(agent_name=AGENT_NAME, session_id=SESSION_B)
print(f"deleted session B ({SESSION_B})\n")

try:
    get_session(SESSION_B)
except ResourceNotFoundError as e:
    print(f"get_session: HTTP 404 {str(e).splitlines()[0]}\n")

print("A call that still carries the deleted session ID:")
try:
    ask("List my notes.", session_id=SESSION_B, retries=1)
    get_session(SESSION_B)
except openai.APIStatusError as e:
    print(f"responses.create: HTTP {e.status_code} {e.message[:200]}")

deleted session B (ba16395922db74380cd1e408726dc68ea012a3dd2b61315cee9dc1e10482daf)

get_session: HTTP 404 (session_not_found) Session '{session-id}' was not found or has been deleted. [Request ID: 440db414683fb0a82e465a6a9bc3fc76]

A call that still carries the deleted session ID:


🧑 List my notes.
🤖 No notes in this session yet.
   agent_session_id={session-id}  response.id=caresp_079f5fc07f9d647b00DQNUAGMSbZfXGaWBorvX9tdDMpMqZ4q1

  ba16395922db74380cd1e408726dc68ea012a3dd2b61315cee9dc1e10482daf  active    created=14:44:02  last_accessed=14:44:02  stopped=-  expires=2026-11-10  v2


## Cleanup

Delete the sessions this run created, including the sandbox that Step 8 recreated under session
B's ID, so repeated runs do not leave sandboxes behind. The agent itself stays; the cleanup cell
in 08-12-01 removes it.

In [11]:
for name, session_id in (("A", SESSION_A), ("B", SESSION_B), ("C", SESSION_C)):
    try:
        client.agents.delete_session(agent_name=AGENT_NAME, session_id=session_id)
        print(f"deleted session {name} ({session_id})")
    except ResourceNotFoundError:
        print(f"session {name} ({session_id}) was already gone")

remaining = list(client.agents.list_sessions(agent_name=AGENT_NAME, limit=100))
print(f"\n{len(remaining)} sessions remain on {AGENT_NAME}, by status: {dict(Counter(label(s.status) for s in remaining))}")

deleted session A (e49b1f8c819c9ccab9bc76d5a0632743e35c187ed6f56f30b77a032732043cc)


deleted session B (ba16395922db74380cd1e408726dc68ea012a3dd2b61315cee9dc1e10482daf)


deleted session C ({session-id})



3 sessions remain on contoso-rm-notes-agent, by status: {'idle': 2, 'deleted': 1}
